# 4. The author's hand: four words, then seventy

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/Meitei-Mayek-Diffusion-model/blob/main/notebooks/4_owner_hand.ipynb)

**Runtime: A100.** About 30 minutes.

The 100 real words of the word-recognition project (written by the author on an iPad) are
split once with `numpy.random.default_rng(2026).permutation(100)`: the first 70 for adaptation,
the first 4 of those the fixed references, the last 30 held out.

1. **Prepare** the real words as one writer (`WORK/data/owner/`) and write the split to
   `results/owner_split.json`.
2. **Adapt** the chosen average for 1,500 steps at lr 2e-5, batches of the author's words
   alternating with synthetic training words (about 15 minutes).
3. **Score** on the 30 held-out words: (a) the main model with the 4 references, (b) the
   adapted model; CER and WER through the recogniser, HWD and KID against the real words; then
   both on 100 validation writers, to check that (b) still writes other hands. A sheet shows
   real | few-shot | adapted.

Inputs: `WORK/runs/main/final.pt` and `last.pt`, `results/eval_val_grid.json` (notebook 3),
`WORK/data`. Outputs: `WORK/runs/owner/owner.pt`, `results/owner.json`, `WORK/eval/owner.png`.

In [ ]:
import os

# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-diffusion"         # this project's working folder
WORK_OLD = "/content/drive/MyDrive/meitei-word-recognition"   # the word-recognition project's folder
REPO = "chingkheinganba231005/Meitei-Mayek-Diffusion-model"
BRANCH = "main"
WORD_REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
WORD_COMMIT = "bdee9e7"                                       # the synthesiser this project builds on
CONTENT = "/content"                                          # the runtime's own disk

HF_REC = "Chingkheinganba/handwritten-meitei-mayek-word-recognition"   # the deployed word recogniser
REC_REVISION = "main"                       # its version is recorded in results/recogniser_source.json
HWD_COMMIT = "eabb5b5"                      # the HWD package (github.com/aimagelab/HWD)

ADAPT_STEPS = 1500
SEEDS = 3              # samples of each held-out text
VAL_WRITERS = 100

DISCONNECT_AT_END = True    # the last cell disconnects the runtime (Colab bills a connected runtime)

In [ ]:
import json, re, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
os.chdir(CONTENT)
CODE, WORD = f"{CONTENT}/code", f"{CONTENT}/word_repo"


def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")


def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails.
    Returns the output."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")
    return "".join(lines)


def clone(url, path, ref):
    """A fresh copy of a repository at a branch or a commit."""
    shutil.rmtree(path, ignore_errors=True)
    run("git", "clone", "-q", url, path)
    run("git", "-C", path, "checkout", "-q", ref)


clone(f"https://github.com/{REPO}.git", CODE, BRANCH)
clone(f"https://github.com/{WORD_REPO}.git", WORD, WORD_COMMIT)
sys.path[:0] = [CODE, WORD]
os.chdir(CODE)

run(sys.executable, "-m", "pip", "install", "-q", "diffusers==0.41.0", "safetensors", "huggingface_hub", "timm")

if not os.path.exists(f"{CONTENT}/HWD/setup.py"):
    clone("https://github.com/aimagelab/HWD.git", f"{CONTENT}/HWD", HWD_COMMIT)
run(sys.executable, "-m", "pip", "install", "-q", f"{CONTENT}/HWD", "gudhi")   # gudhi: imported by hwd.scores

for d in ("data", "runs", "results", "eval"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
run("git", "-C", CODE, "log", "-1", "--format=code: %h %s")

run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")

REC = f"{CONTENT}/recogniser"


def get_recogniser():
    """The deployed recogniser (recogniser.pt, char_lm.pkl, decoding.json) at the version recorded the
    first time (results/recogniser_source.json), so that every notebook scores with the same one."""
    from huggingface_hub import HfApi, snapshot_download
    kept = f"{WORK}/results/recogniser_source.json"
    if os.path.exists(kept):
        source = json.loads(Path(kept).read_text())
    else:
        source = {"repo": HF_REC, "revision": REC_REVISION, "sha": HfApi().model_info(HF_REC, revision=REC_REVISION).sha}
    if not os.path.exists(f"{REC}/source.json"):
        snapshot_download(HF_REC, revision=source["sha"], local_dir=REC,
                          allow_patterns=["recogniser.pt", "char_lm.pkl", "decoding.json", "results.json"])
        Path(f"{REC}/source.json").write_text(json.dumps(source))
    if not os.path.exists(kept):
        shutil.copy(f"{REC}/source.json", kept)
    print("recogniser:", json.loads(Path(f"{REC}/source.json").read_text()))


get_recogniser()

from IPython.display import Image as Show, display
WEIGHTS, GRID = f"{WORK}/runs/main/final.pt", f"{WORK}/results/eval_val_grid.json"
OWNER, ADAPTED = f"{WORK}/data/owner", f"{WORK}/runs/owner/owner.pt"
for f in (WEIGHTS, GRID):
    if not os.path.exists(f):
        raise SystemExit(f"{f} does not exist yet: run notebooks 2 and 3 first")

## 1. Prepare

In [ ]:
stream(sys.executable, "-u", "scripts/owner.py", "prepare", "--word-repo", WORD, "--real", f"{WORD}/real_words",
       "--out", OWNER, "--split", f"{WORK}/results/owner_split.json")

## 2. Adapt

In [ ]:
if not os.path.exists(ADAPTED):
    stream(sys.executable, "-u", "scripts/owner.py", "adapt", "--word-repo", WORD, "--weights", WEIGHTS,
           "--setting", GRID, "--last", f"{WORK}/runs/main/last.pt", "--owner", OWNER, "--data", f"{WORK}/data",
           "--steps", ADAPT_STEPS, "--out", ADAPTED)
print(f"{ADAPTED}: {os.path.getsize(ADAPTED) / 1e6:.0f} MB")

## 3. Score

In [ ]:
stream(sys.executable, "-u", "scripts/owner.py", "score", "--word-repo", WORD, "--weights", WEIGHTS,
       "--adapted", ADAPTED, "--setting", GRID, "--owner", OWNER, "--data", f"{WORK}/data", "--recogniser", REC,
       "--work", f"{CONTENT}/eval/owner", "--out", f"{WORK}/results/owner.json", "--sheet", f"{WORK}/eval/owner.png",
       "--seeds", SEEDS, "--val-writers", VAL_WRITERS)
display(Show(f"{WORK}/eval/owner.png"))   # each row: real | few-shot | adapted

## Disconnect

Colab bills a connected runtime whether or not it is working. The cell below disconnects it
(`DISCONNECT_AT_END = True`); otherwise use *Runtime > Disconnect and delete runtime*. Every
result is already on Drive.

In [ ]:
# Colab bills a connected runtime whether or not it is working: disconnect it now.
if DISCONNECT_AT_END:
    from google.colab import runtime
    runtime.unassign()
else:
    print("Finished. Disconnect the runtime: Runtime > Disconnect and delete runtime.")